In [1]:
import pandas as pd
import numpy as np
from plz_to_bundesland import get_bundesland

In [2]:
car_data = pd.read_excel('../data/raw/fz10_2026_09.xlsx', sheet_name='Sheet1')

car_data = car_data.drop(index=0).reset_index(drop=True)



car_data = car_data[
    (~car_data['Marke'].str.strip().str.endswith('ZUSAMMEN', na=False)) &
    (~car_data['Marke'].str.strip().str.endswith('INSGESAMT', na=False))
].copy()

car_data['Marke'] = car_data['Marke'].ffill()


car_data = car_data.replace('-', 0)

column_mapping = {
    'Marke' : 'brand',
    'Modellreihe': 'model',
    'Insgesamt': 'total_sep_2026',
    'Unnamed: 3': 'total_jan_sep_2026',
    'Unnamed: 4': 'total_share_pct',
    'mit Dieselantrieb': 'diesel_sep_2026',
    "Unnamed: 6": 'diesel_jan_sep_2026',
    'Unnamed: 7': 'diesel_share_pct',
    "mit Hybridantrieb \n(incl. Plug-in-Hybrid)": 'hybrid_incl_phev_sep_2026',
    'Unnamed: 9': 'hybrid_incl_phev_jan_sep_2026',
    'Unnamed: 10': 'hybrid_incl_phev_share_pct',
    'Benzin-Hybridantrieb \n(incl. Plug-in-Hybrid)': 'petrol_hybrid_incl_phev_sep_2026',
    'Unnamed: 12': 'petrol_hybrid_incl_phev_jan_sep_2026',
    'Unnamed: 13': 'petrol_hybrid_incl_phev_share_pct',
    'Diesel-Hybridantrieb \n(incl. Plug-in-Hybrid)': 'diesel_hybrid_incl_phev_sep_2026',
    'Unnamed: 15': 'diesel_hybrid_incl_phev_jan_sep_2026',
    'Unnamed: 16': 'diesel_hybrid_incl_phev_share_pct',
    'Hybridantrieb \n(ohne Plug-in-Hybrid)': 'hybrid_non_plugin_sep_2026',
    'Unnamed: 18': 'hybrid_non_plugin_jan_sep_2026',
    'Unnamed: 19': 'hybrid_non_plugin_share_pct',
    'Benzin-Hybridantrieb \n(ohne Plug-in-Hybrid)': 'petrol_hybrid_non_plugin_sep_2026',
    'Unnamed: 21': 'petrol_hybrid_non_plugin_jan_sep_2026',
    'Unnamed: 22': 'petrol_hybrid_non_plugin_share_pct',
    'Diesel-Hybridantrieb \n(ohne Plug-in-Hybrid)': 'diesel_hybrid_non_plugin_sep_2026',
    'Unnamed: 24': 'diesel_hybrid_non_plugin_jan_sep_2026',
    'Unnamed: 25': 'diesel_hybrid_non_plugin_share_pct',
    'Plug-in-Hybridantrieb': 'phev_sep_2026',
    'Unnamed: 27': 'phev_jan_sep_2026',
    'Unnamed: 28': 'phev_share_pct',
    'Benzin-Plug-in-Hybridantrieb': 'petrol_phev_sep_2026',
    'Unnamed: 30': 'petrol_phev_jan_sep_2026',
    'Unnamed: 31': 'petrol_phev_share_pct',
    'Diesel-Plug-in-Hybridantrieb': 'diesel_phev_sep_2026',
    'Unnamed: 33': 'diesel_phev_jan_sep_2026',
    'Unnamed: 34': 'diesel_phev_share_pct',
    'mit Elektroantrieb (BEV)': 'bev_sep_2026',
    'Unnamed: 36': 'bev_jan_sep_2026',
    'Unnamed: 37': 'bev_share_pct',
    'mit Allradantrieb': 'all_wheel_drive_sep_2026',
    'Unnamed: 39': 'all_wheel_drive_jan_sep_2026',
    'Unnamed: 40': 'all_wheel_drive_share_pct',
    'Cabriolets': 'cabriolets_sep_2026',
    'Unnamed: 42': 'cabriolets_jan_sep_2026',
    'Unnamed: 43': 'cabriolets_share_pct'

}

car_data = car_data.rename(columns=column_mapping, errors='raise')


numeric_cols = car_data.columns.drop(['brand', 'model'])

car_data[numeric_cols] = car_data[numeric_cols].apply(
    pd.to_numeric,
    errors='raise'
)


car_data[car_data['brand'] == 'BMW'].filter(regex='^.*_pct$')

df = car_data.filter(regex=r'^(?!.*_jan_sep_).*$')


df_subset = df[
    (df['brand'] == 'BMW') &
    (
        (df['bev_sep_2026'] > 0) |
        (df['phev_sep_2026'] > 0)
    )
].copy()

df_subset['chargeable_vehicles'] = (
    df_subset['bev_sep_2026'] +
    df_subset['phev_sep_2026']
)

df_subset = df_subset[
    ['brand', 'model', 'bev_sep_2026',
     'phev_sep_2026', 'chargeable_vehicles']
]

print(df_subset.sort_values(by='chargeable_vehicles', ascending=False))

   brand     model  bev_sep_2026  phev_sep_2026  chargeable_vehicles
44   BMW        X1          1809            343                 2152
46   BMW        X3          1072            825                 1897
39   BMW       5ER           523            509                 1032
38   BMW       4ER           777              0                  777
48   BMW        X5             0            370                  370
37   BMW       3ER           140            193                  333
43   BMW        IX           218              0                  218
45   BMW        X2           136              0                  136
40   BMW       7ER            76             13                   89
51   BMW        XM             0             64                   64
36   BMW       2ER             0             32                   32
54   BMW  SONSTIGE             0              2                    2


In [3]:
charging_stations = pd.read_csv('../data/raw/rhein-kreis-neuss-ladesaulen-in-deutschland.csv', sep=';')
charging_stations['Postleitzahl_original'] = (
    charging_stations['Postleitzahl'].copy()
)

plz = charging_stations['Postleitzahl'].astype('string').str.zfill(5)

charging_stations['Bundesland'] = plz.map(
    lambda x: get_bundesland(x) if pd.notna(x) else 'Unknown'
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].replace('Unknown', pd.NA)
)

In [4]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_cities = (
    known.groupby('Ort')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

city_lookup = (
    known[known['Ort'].isin(valid_cities)]
    .drop_duplicates('Ort')
    .set_index('Ort')['Bundesland']
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].fillna(
        charging_stations['Ort'].map(city_lookup)
    )
)

In [5]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_districts = (
    known.groupby('Kreis/kreisfreie Stadt')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

district_lookup = (
    known[known['Kreis/kreisfreie Stadt'].isin(valid_districts)]
    .drop_duplicates('Kreis/kreisfreie Stadt')
    .set_index('Kreis/kreisfreie Stadt')['Bundesland']
)

charging_stations['Bundesland'] = (
    charging_stations['Bundesland'].fillna(
        charging_stations['Kreis/kreisfreie Stadt'].map(district_lookup)
    )
)

In [6]:
remaining = charging_stations[
    charging_stations['Bundesland'].isna()
]

print("Unresolved rows:", len(remaining))

display(
    remaining[
        ['Ort', 'Postleitzahl', 'Kreis/kreisfreie Stadt',
         'Straße', 'Hausnummer', 'Bundesland']
    ]
)

Unresolved rows: 1


,Ort,Postleitzahl,Kreis/kreisfreie Stadt,Straße,Hausnummer,Bundesland
59532,Weimar,9925,Kreisfreie Stadt Weimar,Henry-van-de-Velde-Straße,2,<NA>


In [7]:
charging_stations.at[59532,'Postleitzahl'] = 99425
charging_stations.at[59532,'Bundesland'] = "Thüringen"

In [8]:


charging_stations['Postleitzahl'] = charging_stations['Postleitzahl'].astype(str)
charging_stations.at[1162,'Postleitzahl'] = "06571"
charging_stations.at[1166,'Postleitzahl'] = "07607"
charging_stations.at[1167,'Postleitzahl'] = "07607"
charging_stations.at[1168,'Postleitzahl'] = "07613"
charging_stations.at[1169,'Postleitzahl'] = "07639"


bundeslaender = [
    'Sachsen', 'Sachsen-Anhalt', 'Thüringen', 'Brandenburg'
]

mask = (
    (
        # charging_stations['Ort'].isin(cities) |
        charging_stations['Bundesland'].isin(bundeslaender)
    )
    & (charging_stations['Postleitzahl'].str.len() == 4)
)

bayern_mask = (
    (charging_stations['Bundesland'] == 'Bayern') &
    (charging_stations['Postleitzahl'].str.len() == 4)
)


charging_stations.loc[mask, 'Postleitzahl'] = (
    charging_stations.loc[mask, 'Postleitzahl'].str.zfill(5)
)

charging_stations.loc[bayern_mask, 'Postleitzahl'] = (
    '9' + charging_stations.loc[bayern_mask, 'Postleitzahl']
)

charging_stations.loc[52639, 'Postleitzahl'] = "33102"
charging_stations.loc[28265, 'Postleitzahl'] = "50933"
charging_stations.loc[9035, 'Postleitzahl'] = "53919"
charging_stations.loc[59644, 'Postleitzahl'] = "53919"
charging_stations.loc[37483, 'Postleitzahl'] = "72072"
charging_stations.loc[39172, 'Postleitzahl'] = "72793"
charging_stations.loc[43320, 'Postleitzahl'] = "55444"
charging_stations.loc[30173, 'Postleitzahl'] = "23558"

print(len(charging_stations[charging_stations['Postleitzahl'].astype(str).str.len() < 5]))

0


In [9]:
charging_stations['Ort'] = charging_stations['Ort'].replace(
    'Halle', 'Halle (Saale)'
)

In [10]:
known = charging_stations.dropna(subset=['Bundesland'])

valid_cities = (
    known.groupby('Ort')['Bundesland']
    .nunique()
    .loc[lambda x: x == 1]
    .index
)

city_lookup = (
    known[known['Ort'].isin(valid_cities)]
    .drop_duplicates('Ort')
    .set_index('Ort')['Bundesland']
)

expected = charging_stations['Ort'].map(city_lookup)

mismatches = charging_stations[
    expected.notna()
    & charging_stations['Bundesland'].notna()
    & charging_stations['Bundesland'].ne(expected).fillna(False)
]

display(
    mismatches[
        ['Ort', 'Postleitzahl', 'Bundesland',
         'Kreis/kreisfreie Stadt', 'Straße', 'Hausnummer']
    ]
)

,Ort,Postleitzahl,Bundesland,Kreis/kreisfreie Stadt,Straße,Hausnummer


In [11]:
plz = charging_stations['Postleitzahl'].astype('string')

print("Total records:", len(charging_stations))
print("Missing Bundesland:", charging_stations['Bundesland'].isna().sum())
print("Unique Bundesländer:", charging_stations['Bundesland'].nunique())
print("Missing PLZ:", plz.isna().sum())
print("Invalid PLZ format:", (~plz.str.fullmatch(r'\d{5}').fillna(False)).sum())
print("City/state mismatches:", len(mismatches))

Total records: 61793
Missing Bundesland: 0
Unique Bundesländer: 16
Missing PLZ: 0
Invalid PLZ format: 0
City/state mismatches: 0


In [12]:
from pathlib import Path

output_path = Path('../data/processed')
output_path.mkdir(parents=True, exist_ok=True)

charging_stations.to_csv(
    output_path / 'charging_stations_cleaned.csv',
    index=False
)

In [13]:
data = pd.read_excel('../data/raw/fz28_2026_09.xlsx',
                      sheet_name="FZ 28.4",
                      header=None)


brands = data.iloc[
    12:79,1

]

total_sales = data.iloc[
    12:79,2
    ]

total_electric_sales = data.iloc[
    12:79,7
].replace('-', 0)


electric_sales_pct = (
    total_electric_sales.astype(float) /
    total_sales.astype(float)
)

ev_market_share = (
    total_electric_sales.astype(float) / 
    total_electric_sales.astype(float).sum()
)

score = (np.sqrt(electric_sales_pct * ev_market_share) * 100).round(2)

cleaned_data = pd.DataFrame({
    'brands' : brands.str.lower(),
    'total_sales' : total_sales.astype('int32'),
    'electric_sales': total_electric_sales.astype('int32'),
    'electric_pct': electric_sales_pct,
    'ev_market_share': (ev_market_share * 100).round(3),
    'electric_sales_score': score
}).reset_index(drop=True)

cleaned_data = cleaned_data.sort_values(
    'electric_sales_score',
    ascending=False
).reset_index(drop=True)

cleaned_data


,brands,total_sales,electric_sales,electric_pct,ev_market_share,electric_sales_score
0,tesla,12552,12552,1.000000,14.167,37.64
1,skoda,19898,7673,0.385617,8.660,18.27
2,kia,6125,4184,0.683102,4.722,17.96
3,leapmotor,2652,2570,0.969080,2.901,16.77
4,vw,42461,9440,0.222322,10.655,15.39
5,mercedes,21595,6497,0.300857,7.333,14.85
6,audi,16361,5265,0.321802,5.943,13.83
7,mini,4121,2389,0.579714,2.696,12.50
8,renault,6138,2782,0.453242,3.140,11.93
9,xpeng,1250,1250,1.000000,1.411,11.88


In [14]:
data_per_bundesland = pd.read_excel('../data/raw/fz28_2026_09.xlsx',
                      sheet_name="FZ 28.9",
                      header=None)

regions = data_per_bundesland.iloc[13:30, 1]
total_sales_per_bundesland = data_per_bundesland.iloc[13:30, 2].astype(int)
total_ev_sales_per_bundesland = data_per_bundesland.iloc[13:30, 7].astype(int)


data_per_bundesland_cleaned = pd.DataFrame({
    'bundesland' : regions,
    'total_sales': total_sales_per_bundesland,
    'ev_sales': total_ev_sales_per_bundesland,


}).reset_index(drop=True)

regional_data = data_per_bundesland_cleaned[
    data_per_bundesland_cleaned['bundesland'] != 'Sonstige'
].copy()

print("Total registrations:",
      data_per_bundesland_cleaned['total_sales'].sum())

print("BEV registrations:",
      data_per_bundesland_cleaned['ev_sales'].sum())

print("Sonstige:",
      data_per_bundesland_cleaned.loc[
          data_per_bundesland_cleaned['bundesland'] == 'Sonstige'
      ])


Total registrations: 256774
BEV registrations: 88599
Sonstige:    bundesland  total_sales  ev_sales
16   Sonstige          164        52


In [15]:
print(charging_stations.shape)
print(data_per_bundesland_cleaned.shape)
print(charging_stations['Bundesland'].isna().sum())

(61793, 29)
(17, 3)
0


In [52]:
counts = charging_stations.groupby("Bundesland").size().reset_index(name="count")

counts.sort_values(by='count', ascending=False)

,Bundesland,count
1,Bayern,12113
9,Nordrhein-Westfalen,11913
0,Baden-Württemberg,10486
8,Niedersachsen,6152
6,Hessen,4442
2,Berlin,2695
10,Rheinland-Pfalz,2586
12,Sachsen,2405
14,Schleswig-Holstein,2252
3,Brandenburg,1395


In [ ]:
connector_cols = [f'Steckertypen{i}' for i in range(1, 7)]

charging_stations_by_type = charging_stations.copy()

charging_stations_by_type['has_AC'] = (
    charging_stations_by_type[connector_cols]
    .apply(lambda col: col.str.contains('AC', na=False))
    .any(axis=1)
)

charging_stations_by_type['has_DC'] = (
    charging_stations_by_type[connector_cols]
    .apply(lambda col: col.str.contains('DC', na=False))
    .any(axis=1)
)

charging_stations_by_type


In [ ]:
charging_types_by_bundesland = (charging_stations_by_type.groupby('Bundesland')[['has_AC', 'has_DC']].sum())

charging_types_by_bundesland

In [60]:
charging_stations_by_type.to_parquet('../data/processed/charging_stations_clean.parquet', index=False)

In [43]:
merged_data = data_per_bundesland_cleaned.merge(
    charging_types_by_bundesland,
    left_on='bundesland',
    right_index=True,
    validate='one_to_one'
)

merged_data

,bundesland,total_sales,ev_sales,has_AC,has_DC
0,Baden-Württemberg,34959,12088,9509,1763
1,Bayern,51082,18238,10519,2496
2,Berlin,4315,1576,2475,345
3,Brandenburg,4421,1674,1123,425
4,Bremen,1435,486,384,77
5,Hamburg,10905,1991,1109,238
6,Hessen,28073,7625,3852,926
7,Mecklenburg-Vorpommern,2621,856,743,223
8,Niedersachsen,30052,11213,5237,1562
9,Nordrhein-Westfalen,52713,19696,10411,2579
